# Text Classification — Spam Detection

**Goal:** Train a machine learning model to classify messages as spam or not spam.

---

## Pipeline

```
Raw text  →  Vectorize (word counts)  →  Train Logistic Regression  →  Predict
```

### Step 1 — CountVectorizer
Converts each message into a **bag-of-words** vector:
```
Vocab: [claim, earn, free, good, how, meet, money, morning, ...]

"free ticket win"  →  [0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1]
                                ↑free                   ↑ticket  ↑win
```

### Step 2 — Logistic Regression
Learns weights for each word. High weight = strong spam indicator.

```
P(spam) = sigmoid(w₁×free + w₂×win + w₃×earn + ...)
```

- Output > 0.5 → Spam (label = 1)
- Output ≤ 0.5 → Ham  (label = 0)

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

# ── Dataset ────────────────────────────────────────────────
messages = [
    ("free ticket win",   1),   # spam
    ("good morning",      0),   # ham
    ("earn money online", 1),   # spam
    ("how are you",       0),   # ham
    ("claim your prize",  1),   # spam
    ("nice to meet you",  0),   # ham
]

df = pd.DataFrame(messages, columns=["message", "label"])

print("Training Data:")
print(df.to_string(index=False))
print()
print("Class distribution:")
print(df["label"].value_counts().rename({0: "Ham (0)", 1: "Spam (1)"}))


Training Data:
          message  label
  free ticket win      1
     good morning      0
earn money online      1
      how are you      0
 claim your prize      1
 nice to meet you      0

Class distribution:
label
Spam (1)    3
Ham (0)     3
Name: count, dtype: int64


In [2]:
# ── Step 1: Vectorize text ────────────────────────────────
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df["message"])
y = df["label"]

print("Vocabulary learned:")
print(vectorizer.get_feature_names_out())
print()
print("Feature matrix (word counts per message):")
print(pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out()).to_string())


Vocabulary learned:
['are' 'claim' 'earn' 'free' 'good' 'how' 'meet' 'money' 'morning' 'nice'
 'online' 'prize' 'ticket' 'to' 'win' 'you' 'your']

Feature matrix (word counts per message):
   are  claim  earn  free  good  how  meet  money  morning  nice  online  prize  ticket  to  win  you  your
0    0      0     0     1     0    0     0      0        0     0       0      0       1   0    1    0     0
1    0      0     0     0     1    0     0      0        1     0       0      0       0   0    0    0     0
2    0      0     1     0     0    0     0      1        0     0       1      0       0   0    0    0     0
3    1      0     0     0     0    1     0      0        0     0       0      0       0   0    0    1     0
4    0      1     0     0     0    0     0      0        0     0       0      1       0   0    0    0     1
5    0      0     0     0     0    0     1      0        0     1       0      0       0   1    0    1     0


In [3]:
# ── Step 2: Train Logistic Regression ─────────────────────
model = LogisticRegression()
model.fit(X, y)

# Show learned word weights
weights = pd.Series(
    model.coef_[0],
    index=vectorizer.get_feature_names_out()
).sort_values(ascending=False)

print("Word weights (positive = spam indicator, negative = ham indicator):")
print(weights.round(3).to_string())


Word weights (positive = spam indicator, negative = ham indicator):
claim      0.285
your       0.285
earn       0.285
free       0.285
money      0.285
win        0.285
ticket     0.285
prize      0.285
online     0.285
to        -0.238
nice      -0.238
meet      -0.238
are       -0.271
how       -0.271
good      -0.347
morning   -0.347
you       -0.509


In [4]:
# ── Step 3: Predict new messages ──────────────────────────
test_messages = [
    "win free ticket",
    "how are you doing",
    "claim money now",
    "good afternoon"
]

for msg in test_messages:
    vec  = vectorizer.transform([msg])
    pred = model.predict(vec)[0]
    prob = model.predict_proba(vec)[0]

    label = "🚫 SPAM" if pred == 1 else "✅ HAM"
    print(f"{msg:<25}  →  {label}  (spam prob: {prob[1]:.2f})")


win free ticket            →  🚫 SPAM  (spam prob: 0.71)
how are you doing          →  ✅ HAM  (spam prob: 0.27)
claim money now            →  🚫 SPAM  (spam prob: 0.65)
good afternoon             →  ✅ HAM  (spam prob: 0.43)


## Key Concepts

| Concept | What it does |
|---------|-------------|
| `CountVectorizer` | Converts text to word-count vectors |
| `fit_transform` | Learns vocabulary + transforms training data |
| `transform` (not fit_transform) | Apply learned vocab to new test data |
| `predict_proba` | Returns probability for each class, not just the label |

## What to Try Next
- Add `TfidfVectorizer` in place of `CountVectorizer` — does accuracy change?
- Use a larger dataset (SMS Spam Collection on Kaggle)
- Try `MultinomialNB` (Naive Bayes) — the classic spam filter algorithm
- Add `train_test_split` and report actual accuracy